In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)


In [2]:
train = "train-00000-of-00001-d5675d98fde8c367"

input_file = f"raw/ContractNLI_reuben/{train}.parquet"
df = pd.read_parquet(input_file)
df = df[df['label'] != 'notmentioned']

# df.to_json(json_file, orient='records', lines=True)

In [39]:
df.dtypes

document_id        int64
file_name            str
text                 str
hypothesis_id        str
hypothesis           str
label                str
evidence_spans    object
evidence_texts    object
dtype: object

In [25]:
df.label.value_counts()

label
entailment       3530
contradiction     841
Name: count, dtype: int64

In [33]:
df.shape

(4371, 8)

In [3]:
full_document = df.groupby(['document_id']).nth(0)[['document_id', 'text']]
full_document.head(1)

document_id  \
1           34   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [4]:
evidence_list = df.head(1)['evidence_texts'].values[0].tolist()
# Turn evidence into a single string separated by newlines
evidence_list_str = "\n".join(evidence_list)
hypothesis = df.head(1)['hypothesis'].values[0]
hypo_label = df.head(1)['label'].values[0]
{'hypothesis': hypothesis,
'evidence': evidence_list_str,
'hypothesis_label': hypo_label,}


{'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.',
 'evidence': '5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: \n(a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or ',
 'hypothesis_label': 'entailment'}

In [ ]:
def hypo_inferred(row):
    # Turn evidence into a single string separated by newlines
    evidence_list = row['evidence_texts'].tolist()
    evidence_list_str = "\n".join(evidence_list)

    hypothesis = row['hypothesis']
    hypo_label = row['label']

    data_dict = {'hypothesis': hypothesis,
            'evidence': evidence_list_str,
            'hypothesis_label': hypo_label}
    
    return str(data_dict)

hypotheses_inferred = df.copy()[['document_id','evidence_texts','hypothesis','label']]
hypotheses_inferred['inference'] = hypotheses_inferred.apply(hypo_inferred, axis=1)
hypotheses_inferred = hypotheses_inferred[['document_id', 'inference']]

In [17]:
hypotheses_inferred_byid = hypotheses_inferred.groupby('document_id')['inference'].agg(list).reset_index()
hypotheses_inferred_byid.head(1)

document_id  \
0           34   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [18]:
final = pd.merge(full_document, hypotheses_inferred_byid, on='document_id', how='inner')
final.shape

(423, 3)

In [24]:
filename = "train"

output_file = f"processed/{filename}.jsonl"

final.to_json(output_file, orient='records', lines=True)

In [27]:
def process_dataset(input_file, output_file):
    df = pd.read_parquet(input_file)
    df = df[df['label'] != 'notmentioned']

    # full document
    full_document = df.groupby(['document_id']).nth(0)[['document_id', 'text']]

    # hypotheses

    def hypo_inferred(row):
        """
        Helper function
        """
        # Turn evidence into a single string separated by newlines
        evidence_list = row['evidence_texts'].tolist()
        evidence_list_str = "\n".join(evidence_list)

        hypothesis = row['hypothesis']
        hypo_label = row['label']

        data_dict = {'hypothesis': hypothesis,
                'evidence': evidence_list_str,
                'hypothesis_label': hypo_label}
        # return the dictionary as a string
        return str(data_dict)

    hypotheses_inferred = df.copy()[['document_id','evidence_texts','hypothesis','label']]
    hypotheses_inferred['inference'] = hypotheses_inferred.apply(hypo_inferred, axis=1)
    hypotheses_inferred = hypotheses_inferred[['document_id', 'inference']]

    # aggregate the hypo_infer strings into a list by document_id
    hypotheses_inferred_byid = hypotheses_inferred.groupby('document_id')['inference'].agg(list).reset_index()

    # merge full document, and processed hypotheses columns
    # columns are now: document_id, text, inference (str of long dictionary)
    final_df = pd.merge(full_document, hypotheses_inferred_byid, on='document_id', how='inner')

    # write to new location
    final_df.to_json(output_file, orient="records", lines=True)
    print(f"{input_file} processed and written to {output_file}")



In [28]:
input_filename = "train-00000-of-00001-d5675d98fde8c367"
input_file = f"raw/ContractNLI_reuben/{input_filename}.parquet"

output_filename = "train"
output_file = f"processed/{output_filename}.jsonl"

process_dataset(input_file, output_file)

raw/ContractNLI_reuben/train-00000-of-00001-d5675d98fde8c367.parquet processed and written to processed/train.jsonl


In [29]:
input_filename = "test-00000-of-00001-762092f06e2d7b48"
input_file = f"raw/ContractNLI_reuben/{input_filename}.parquet"

output_filename = "test"
output_file = f"processed/{output_filename}.jsonl"

process_dataset(input_file, output_file)

raw/ContractNLI_reuben/test-00000-of-00001-762092f06e2d7b48.parquet processed and written to processed/test.jsonl


In [30]:
input_filename = "validation-00000-of-00001-3714eab8cedda2e5"
input_file = f"raw/ContractNLI_reuben/{input_filename}.parquet"

output_filename = "validation"
output_file = f"processed/{output_filename}.jsonl"

process_dataset(input_file, output_file)

raw/ContractNLI_reuben/validation-00000-of-00001-3714eab8cedda2e5.parquet processed and written to processed/validation.jsonl
